In [2]:
import pandas as pd
from snownlp import SnowNLP
import json

## Tools: Format Results

In [3]:
categories = [
    '政治与政策', 
    '经济与金融', 
    '文化与旅游', 
    '体育赛事', 
    '教育与人才', 
    '医疗与公共健康', 
    '环境与可持续发展', 
    '交通与基础设施', 
    '法律与治安管理', 
    '应急与安全', 
    '科技与创新',
    '就业与培训',
    '民生与便民服务',
    '其他'
]


In [4]:
def extract_info(file_path):
    ds = []  
    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                item = json.loads(line.strip())
                _id = item.get('custom_id') 
                text = item.get('response').get('body').get('choices')[0].get('message').get('content')
                if text:
                    text = text.strip()
                    text = text.replace('。', '')
                    result = []
                    for c in categories:
                        if c in text:
                            result.append(c)
                    text = "@".join(result)
                    ds.append([_id, text])
            except json.JSONDecodeError:
                print(f'无法解析文件 {file_path} 中的一行: {line.strip()}')

    temp = pd.DataFrame(ds, columns=['idx', 'label'])
    return temp

In [13]:
## task 1
local_res = extract_info('./data/llm-gen-class-results/local_cls_task_success.jsonl')
local_res['idx'] = local_res['idx'].astype(int)
local_res

,idx,label
0,0,体育赛事
1,1,文化与旅游@应急与安全
2,8,经济与金融@文化与旅游
3,3,文化与旅游@应急与安全
4,4,文化与旅游@交通与基础设施@应急与安全
...,...,...
2063,2062,文化与旅游@环境与可持续发展@交通与基础设施@民生与便民服务
2064,2067,民生与便民服务
2065,2065,医疗与公共健康
2066,915,政治与政策


In [14]:
local_news = pd.read_csv('./data/woa_macau.csv')
local_news['idx'] = local_news.index

# merge
local = pd.merge(local_news, local_res, on='idx')
local.to_csv('./data/topic-results/local_topic.csv', index=None)

In [16]:
## task 2
non_local_res = extract_info('./data/llm-gen-class-results/non_local_cls_task_success.jsonl')
non_local_res['idx'] = non_local_res['idx'].astype(int)

non_local_res.to_csv('./data/topic-results/non_local_topic.csv', index=None)
non_local_res

,idx,label
0,1,政治与政策@经济与金融@文化与旅游
1,0,政治与政策@经济与金融@文化与旅游
2,5,政治与政策
3,2,政治与政策
4,3,政治与政策
...,...,...
4299,4303,政治与政策
4300,4300,政治与政策
4301,4301,经济与金融@科技与创新
4302,4302,政治与政策


In [17]:
non_local_news = pd.read_csv('./data/woa_other_cities.csv')
non_local_news['idx'] = non_local_news.index

# merge
non_local = pd.merge(non_local_news, non_local_res, on='idx')
non_local.to_csv('./data/topic-results/non_local_topic.csv', index=None)